In [1]:
# Cell 1: FIRST RUN ONLY - Clone & Install Everything
# After running this cell, RESTART KERNEL, then SKIP to Cell 2

import os
import shutil

%cd /workspace

# Clean slate
if os.path.exists("InternVL"):
    print("Removing old InternVL directory...")
    shutil.rmtree("InternVL")

# Clone and setup
print("Cloning Blind-Assist/InternVL...")
!git clone https://github.com/Blind-Assist/InternVL.git
%cd InternVL
!git checkout walkvlm
%cd internvl_chat

# Install ALL dependencies in one go
print("\n📦 Installing dependencies...")
!pip install -q \
    "transformers==4.37.2" \
    "peft==0.10.0" \
    "accelerate<1" \
    "deepspeed>=0.13.5" \
    "timm==0.9.12" \
    "einops==0.6.1" \
    "sentencepiece==0.1.99" \
    "tokenizers==0.15.1" \
    "datasets" \
    "huggingface_hub" \
    "decord" \
    "wandb" \
    "opencv-python-headless" \
    "numpy==1.26.4" \
    "scipy" \
    "scikit-learn>=1.2.2" \
    "orjson" \
    "pyyaml" \
    "termcolor" \
    "yacs"

# Install bitsandbytes (CUDA 12.x compatible version)
!pip uninstall bitsandbytes -y 2>/dev/null || true
!pip install -q bitsandbytes>=0.43.0

# Install flash-attention
!pip install -q flash-attn --no-build-isolation

# Fix typing_extensions
!pip install -q --upgrade "typing_extensions>=4.12.0" pydantic pydantic-core

print("\n" + "="*50)
print("✅ Installation complete!")
print("="*50)
print("⚠️  NOW: Restart Kernel (Kernel → Restart)")
print("⚠️  THEN: Skip this cell, run Cell 2")
print("="*50)

/workspace
Cloning Blind-Assist/InternVL...
Cloning into 'InternVL'...


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


remote: Enumerating objects: 3537, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 3537 (delta 117), reused 110 (delta 70), pack-reused 3343 (from 2)
Receiving objects: 100% (3537/3537), 75.96 MiB | 23.26 MiB/s, done.
Resolving deltas: 100% (2113/2113), done.
Updating files: 100% (904/904), done.
/workspace/InternVL
Branch 'walkvlm' set up to track remote branch 'walkvlm' from 'origin'.
Switched to a new branch 'walkvlm'
/workspace/InternVL/internvl_chat

📦 Installing dependencies...

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To updat

In [1]:
# Cell 2: POST-RESTART - Setup Environment
# Run this AFTER kernel restart

import os
%cd /workspace/InternVL/internvl_chat

# Verify environment
print("📁 Current directory:", os.getcwd())
print("\n🔍 Checking installations...")

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ CUDA version: {torch.version.cuda}")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

import bitsandbytes
print(f"✅ bitsandbytes: {bitsandbytes.__version__}")

import transformers
print(f"✅ transformers: {transformers.__version__}")

print("\n✅ Environment ready! Continue to next cell.")

/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/InternVL/internvl_chat
📁 Current directory: /workspace/InternVL/internvl_chat

🔍 Checking installations...
✅ PyTorch: 2.4.1+cu124
✅ CUDA available: True
✅ CUDA version: 12.4
✅ GPU: NVIDIA A40
✅ bitsandbytes: 0.49.2
✅ transformers: 4.37.2

✅ Environment ready! Continue to next cell.


In [2]:
# Cell 3: Login to Services
import wandb
from huggingface_hub import login

print("🔑 Login to WandB:")
wandb.login()

print("\n🔑 Login to Hugging Face:")
login()

🔑 Login to WandB:


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rpgkr3 (vlm-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



🔑 Login to Hugging Face:


In [3]:
# Cell 4: Patch dist_utils.py
patch_code = """import os
import torch
import torch.distributed as dist

def init_dist(launcher='pytorch', backend='nccl', **kwargs):
    if 'SLURM_PROCID' in os.environ:
        _init_dist_slurm(backend, **kwargs)
    else:
        _init_dist_pytorch(backend, **kwargs)

def _init_dist_pytorch(backend, **kwargs):
    rank = int(os.environ.get('RANK', 0))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    
    torch.cuda.set_device(local_rank)
    
    if not dist.is_initialized():
        dist.init_process_group(
            backend=backend,
            init_method='env://',
            world_size=world_size,
            rank=rank
        )
    return local_rank

def _init_dist_slurm(backend, port=None, **kwargs):
    proc_id = int(os.environ['SLURM_PROCID'])
    ntasks = int(os.environ['SLURM_NTASKS'])
    node_list = os.environ['SLURM_NODELIST']
    num_gpus = torch.cuda.device_count()
    torch.cuda.set_device(proc_id % num_gpus)
    
    import subprocess
    addr = subprocess.getoutput(f'scontrol show hostname {node_list} | head -n1')
    
    if port is not None:
        os.environ['MASTER_PORT'] = str(port)
    elif 'MASTER_PORT' not in os.environ:
        os.environ['MASTER_PORT'] = '29500'
    
    os.environ['MASTER_ADDR'] = addr
    os.environ['WORLD_SIZE'] = str(ntasks)
    os.environ['RANK'] = str(proc_id)
    
    dist.init_process_group(backend=backend)
    return proc_id % num_gpus
"""


with open('internvl/dist_utils.py', 'w') as f:
    f.write(patch_code)

print("✅ Patched internvl/dist_utils.py")

✅ Patched internvl/dist_utils.py


In [4]:
# Cell 5: Prepare Data
DATASET_LIMIT = 8500  # Change this as needed

!python prepare_walk_data.py --limit {DATASET_LIMIT}

# Verify data
print("\n📊 Data verification:")
!echo "Train samples:" && wc -l < data/walk_vlm/walk_train.jsonl
!echo "Val samples:" && wc -l < data/walk_vlm/walk_val.jsonl
!echo "\nSample images:" && ls data/walk_vlm/images | head -5

Loading dataset: blind-assist/walk-train (Streaming Mode)...
Resolving data files: 100%|██████████████| 8582/8582 [00:00<00:00, 41272.54it/s]
⚠️ LIMITING dataset to first 8500 videos.
📥 Collecting video metadata...
Scanning videos: 100%|█████████████████████| 8500/8500 [00:12<00:00, 660.24it/s]
✅ Found 8500 valid videos
📊 Video split: 7650 train videos, 850 validation videos

🎬 Processing TRAIN videos...
Train videos:  27%|██████▏                | 2074/7650 [37:20<1:54:57,  1.24s/it][mov,mp4,m4a,3gp,3g2,mj2 @ 0x8da08c0] moov atom not found
[14:35:15] /github/workspace/src/video/video_reader.cc:83: ERROR opening: temp_808.mp4, Invalid data found when processing input
Error extracting frames from video 808: Error reading temp_808.mp4...
Train videos:  54%|███████████▍         | 4166/7650 [1:15:47<1:10:42,  1.22s/it][mov,mp4,m4a,3gp,3g2,mj2 @ 0x8d52540] moov atom not found
[15:13:42] /github/workspace/src/video/video_reader.cc:83: ERROR opening: temp_2069.mp4, Invalid data found when proc

In [5]:
# Cell 6: Cleanup & Check Disk Space
!rm -rf work_dirs/internvl2_5_4b_walk_lora/checkpoint-* 2>/dev/null || true
!rm -rf work_dirs/internvl2_5_4b_walk_lora/tmp-checkpoint-* 2>/dev/null || true
!rm -rf wandb/ 2>/dev/null || true

print("💾 Disk space:")
!df -h /workspace

💾 Disk space:
Filesystem                   Size  Used Avail Use% Mounted on
mfs#eu-se-1.runpod.net:9421  686T  520T  166T  76% /workspace


In [6]:
!pip install "accelerate<1" \
    "bitsandbytes==0.42.0" \
    "decord" \
    "deepspeed>=0.13.5" \
    "einops==0.6.1" \
    "einops-exts==0.0.4" \
    "huggingface_hub" \
    "imageio" \
    "numpy==1.26.4" \
    "opencv-python-headless" \
    "orjson" \
    "peft==0.10.0" \
    "pycocoevalcap" \
    "pyyaml" \
    "scikit-learn>=1.2.2" \
    "scipy" \
    "sentencepiece==0.1.99" \
    "shortuuid" \
    "tensorboardX" \
    "termcolor" \
    "timm==0.9.12" \
    "tokenizers==0.15.1" \
    "transformers==4.37.2" \
    "yacs" \
    "wandb"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 100.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 193.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.49.2
    Uninstalling bitsandbytes-0.49.2:
      Successfully uninstalled bitsandbytes-0.49.2

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
# Cell: Fix bitsandbytes for CUDA 12.x
!pip uninstall bitsandbytes -y
!pip install bitsandbytes>=0.43.0 --upgrade

# Verify installation
!python -c "import bitsandbytes; print('✅ bitsandbytes version:', bitsandbytes.__version__)"
!python -c "import torch; print('✅ CUDA available:', torch.cuda.is_available()); print('✅ CUDA version:', torch.version.cuda)"

Found existing installation: bitsandbytes 0.42.0
Uninstalling bitsandbytes-0.42.0:
  Successfully uninstalled bitsandbytes-0.42.0

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ bitsandbytes version: 0.49.2
✅ CUDA available: True
✅ CUDA version: 12.4


In [8]:
# Cell: Patch Early Stopping into Training Script
%cd /workspace/InternVL/internvl_chat

# Read the file
with open('internvl/train/internvl_chat_finetune.py', 'r') as f:
    content = f.read()

# Check if already patched
if 'EarlyStoppingCallback' in content:
    print("✅ Early stopping already patched!")
else:
    # 1. Add import
    content = content.replace(
        "from transformers import (AutoConfig, AutoModelForCausalLM, AutoTokenizer,\n                          HfArgumentParser, Trainer, TrainingArguments,\n                          set_seed)",
        "from transformers import (AutoConfig, AutoModelForCausalLM, AutoTokenizer,\n                          HfArgumentParser, Trainer, TrainingArguments,\n                          set_seed, EarlyStoppingCallback)"
    )
    
    # 2. Add callback before Trainer
    old_trainer = """    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset if training_args.do_train else None,
        eval_dataset=eval_dataset,  # CHANGED FROM None to eval_dataset
        tokenizer=tokenizer,
        data_collator=collator,
    )"""
    
    new_trainer = """    # Setup early stopping callback
    callbacks = []
    if getattr(training_args, 'load_best_model_at_end', False) and eval_dataset is not None:
        early_stopping_callback = EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.005
        )
        callbacks.append(early_stopping_callback)
        logger.info('✅ Early stopping enabled: patience=3, threshold=0.005')

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset if training_args.do_train else None,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=collator,
        callbacks=callbacks if callbacks else None,
    )"""
    
    content = content.replace(old_trainer, new_trainer)
    
    # Write back
    with open('internvl/train/internvl_chat_finetune.py', 'w') as f:
        f.write(content)
    
    print("✅ Early stopping patched successfully!")

/workspace/InternVL/internvl_chat
✅ Early stopping patched successfully!


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
# Cell 7: Run Training
!chmod +x train_walk_lora.sh
!./train_walk_lora.sh 2>&1 | tee training_log.txt

🚀 InternVL3-2B LoRA Training (Early Stopping)
⚙️  Early Stopping Config:
   patience: 3 evaluations
   threshold: 0.5% improvement
   eval_steps: 15
   epochs: 3 (max)
petrel_client is not installed. If you read data locally instead of from ceph, ignore it.
petrel_client is not installed. Using PIL to load images.
df: /root/.triton/autotune: No such file or directory
02/23/2026 16:36:39 - WARNING - __main__ - Process rank: 0, device: cuda:0, n_gpu: 1distributed training: True, 16-bits training: False
02/23/2026 16:36:39 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=4,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=

In [10]:
# Cell 8: Upload Model to HuggingFace
!python upload_to_hf.py

🔄 Converting checkpoint to adapter format (with all weights)...
   Loading: model.safetensors
   📊 Total checkpoint tensors: 1077
   📊 LoRA tensors: 392
   📊 Other tensors: 405
   📊 Total adapter tensors: 797
   📊 Target modules: {'k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj'}
   📊 LoRA rank: 128
✅ Saved adapter_config.json
✅ Saved adapter_model.safetensors
✅ Saved README.md

📤 Creating repo: blind-assist/internvl3-2b-walk-lora-Epoch3-8500-v2
📤 Uploading to blind-assist/internvl3-2b-walk-lora-Epoch3-8500-v2...
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|▎             | 33.5MB / 1.85GB            

Processing Files (0 / 1)      :   2%|▎             | 33.5MB / 1.85GB, 23.9MB/s  

Processing Files (0 / 1)      :   4%|▌             | 67.1MB / 1.85GB, 41.9MB/s  

Processing Files (0 / 1)      :   5%|▊       

In [11]:
# Cell 10: Verify Videos
%cd /workspace/InternVL/internvl_chat

# Check what's in my_videos folder
print("📹 Videos in my_videos folder:")
!ls -la my_videos/

# Count videos
print("\n📊 Total video files:")
!ls my_videos/*.mp4 my_videos/*.avi my_videos/*.mov 2>/dev/null | wc -l

/workspace/InternVL/internvl_chat
📹 Videos in my_videos folder:
total 42108
drwxrwxrwx  2 root root 2003633 Feb 23 13:52 .
drwxrwxrwx 12 root root 3005961 Feb 24 02:12 ..
-rw-rw-rw-  1 root root  759376 Feb 23 13:52 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
-rw-rw-rw-  1 root root 3677174 Feb 23 13:52 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4
-rw-rw-rw-  1 root root 2372286 Feb 23 13:52 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4
-rw-rw-rw-  1 root root 1871479 Feb 23 13:52 20240918-youtube_short_1aec7c3e80c13cbb75f75f72333148bf_2min24s.mp4
-rw-rw-rw-  1 root root 2584601 Feb 23 13:52 20240918-youtube_short_1cb8f8832a143fde640ff92d7c656280_4m28s.mp4
-rw-rw-rw-  1 root root 2769889 Feb 23 13:52 20240918-youtube_short_2711f140bac74f83aa0a56272ec7f6de_3s.mp4
-rw-rw-rw-  1 root root 3417858 Feb 23 13:52 20240918-youtube_short_83bc88612ff6a00628af923c9ff461cb_2m9s.mp4
-rw-rw-rw-  1 root root 2418653 Feb 23 13:52 20240918-youtube_sh

In [12]:
# Cell 11: Verify Setup Before Testing
%cd /workspace/InternVL/internvl_chat

# Check if test script exists
print("📄 Test script:")
!ls -la test_finetuned_model.py 2>/dev/null || echo "❌ Script not found!"

# Check if videos exist
print("\n📹 Videos in my_videos folder:")
!ls -la my_videos/ 2>/dev/null || echo "❌ my_videos folder not found!"

# Count videos
print("\n📊 Total videos:")
!find my_videos/ -name "*.mp4" -o -name "*.avi" -o -name "*.mov" 2>/dev/null | wc -l

/workspace/InternVL/internvl_chat
📄 Test script:
-rw-rw-rw- 1 root root 37084 Feb 23 13:52 test_finetuned_model.py

📹 Videos in my_videos folder:
total 42108
drwxrwxrwx  2 root root 2003633 Feb 23 13:52 .
drwxrwxrwx 12 root root 3005961 Feb 24 02:12 ..
-rw-rw-rw-  1 root root  759376 Feb 23 13:52 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
-rw-rw-rw-  1 root root 3677174 Feb 23 13:52 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4
-rw-rw-rw-  1 root root 2372286 Feb 23 13:52 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4
-rw-rw-rw-  1 root root 1871479 Feb 23 13:52 20240918-youtube_short_1aec7c3e80c13cbb75f75f72333148bf_2min24s.mp4
-rw-rw-rw-  1 root root 2584601 Feb 23 13:52 20240918-youtube_short_1cb8f8832a143fde640ff92d7c656280_4m28s.mp4
-rw-rw-rw-  1 root root 2769889 Feb 23 13:52 20240918-youtube_short_2711f140bac74f83aa0a56272ec7f6de_3s.mp4
-rw-rw-rw-  1 root root 3417858 Feb 23 13:52 20240918-youtube_short_83bc88612ff6a00628af923c

In [13]:
!pip install pandas openpyxl


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [14]:
# Cell 12: Run Test
!python test_finetuned_model.py --input ./my_videos --output ./inference_results_finetuned

🔄 Loading base model: OpenGVLab/InternVL3-2B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
configuration_internvl_chat.py: 4.04kB [00:00, 7.63MB/s]
configuration_intern_vit.py: 5.55kB [00:00, 7.77MB/s]
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL3-2B:
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL3-2B:
- configuration_internvl_chat.py
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of

In [15]:
# Cell: First check what's in the checkpoint folder
%cd /workspace/InternVL/internvl_chat

print("📁 Main output directory:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/

print("\n📁 Checkpoint-50 directory:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/checkpoint-50/

print("\n🔍 Looking for adapter_config.json:")
!find work_dirs/ -name "adapter_config.json" 2>/dev/null

/workspace/InternVL/internvl_chat
📁 Main output directory:
ls: cannot access 'work_dirs/internvl2_5_4b_walk_lora/': No such file or directory

📁 Checkpoint-50 directory:
ls: cannot access 'work_dirs/internvl2_5_4b_walk_lora/checkpoint-50/': No such file or directory

🔍 Looking for adapter_config.json:
work_dirs/internvl3_2b_walk_lora_peft_v2/adapter_config.json


In [16]:
# Cell: Test with BASE model first (should work correctly)
!python test_finetuned_model.py --input ./my_videos --output ./inference_results_base --base

🔄 Loading base model: OpenGVLab/InternVL3-2B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
✅ Base model loaded!

📁 Input folder: ./my_videos
🚀 Processing 15 videos

📦 Processing 1/15: 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
   ✅ Time: 1.66s
   📝 To the immediate right of the u

In [17]:
# Cell 10: Check Local Trained Model
%cd /workspace/InternVL/internvl_chat

print("📁 Checking local model files:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/

print("\n📄 Check for adapter_config.json (LoRA) or model files (merged):")
!cat work_dirs/internvl2_5_4b_walk_lora/adapter_config.json 2>/dev/null && echo "✅ This is a LoRA adapter" || echo "❌ No adapter_config.json"
!ls work_dirs/internvl2_5_4b_walk_lora/*.safetensors 2>/dev/null && echo "✅ Has safetensors files" || echo "❌ No safetensors"

print("\n📊 Training results:")
!cat work_dirs/internvl2_5_4b_walk_lora/train_results.json 2>/dev/null || echo "No train_results.json"

/workspace/InternVL/internvl_chat
📁 Checking local model files:
ls: cannot access 'work_dirs/internvl2_5_4b_walk_lora/': No such file or directory

📄 Check for adapter_config.json (LoRA) or model files (merged):
❌ No adapter_config.json
❌ No safetensors

📊 Training results:
No train_results.json


In [19]:
!python test_from_huggingface.py --input ./my_videos

🔄 Loading base model: OpenGVLab/InternVL3-2B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
🔄 Downloading LoRA adapter from: blind-assist/internvl3-2b-walk-lora-Epoch3-8500-v2
adapter_model.safetensors: 100%|████████████| 1.85G/1.85G [00:14<00:00, 128MB/s]
adapter_config.json: 100%|█████████████████████| 523/523 [00:00<00:00, 1.38MB/s]
   📊 LoRA rank: 128
   📊 

In [28]:
# Cell: Check v2 adapter weight keys
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

adapter_path = hf_hub_download("blind-assist/internvl3-2b-walk-lora-v2", "adapter_model.safetensors")
weights = load_file(adapter_path)

lora_keys = [k for k in weights.keys() if '.lora_' in k]
other_keys = [k for k in weights.keys() if '.lora_' not in k]

print(f"📊 Total weights: {len(weights)}")
print(f"📊 LoRA weights: {len(lora_keys)}")
print(f"📊 Other weights: {len(other_keys)}")

print("\n📄 Sample LoRA keys:")
for k in lora_keys[:3]:
    print(f"   {k}")

print("\n📄 Sample OTHER keys:")
for k in other_keys[:10]:
    print(f"   {k}")

📊 Total weights: 797
📊 LoRA weights: 392
📊 Other weights: 405

📄 Sample LoRA keys:
   base_model.model.language_model.model.layers.0.mlp.down_proj.lora_A.weight
   base_model.model.language_model.model.layers.0.mlp.down_proj.lora_B.weight
   base_model.model.language_model.model.layers.0.mlp.gate_proj.lora_A.weight

📄 Sample OTHER keys:
   base_model.model.language_model.lm_head.weight
   base_model.model.language_model.model.embed_tokens.weight
   base_model.model.language_model.model.layers.0.input_layernorm.weight
   base_model.model.language_model.model.layers.0.post_attention_layernorm.weight
   base_model.model.language_model.model.layers.1.input_layernorm.weight
   base_model.model.language_model.model.layers.1.post_attention_layernorm.weight
   base_model.model.language_model.model.layers.10.input_layernorm.weight
   base_model.model.language_model.model.layers.10.post_attention_layernorm.weight
   base_model.model.language_model.model.layers.11.input_layernorm.weight
   base_m

In [21]:
!python upload_full_checkpoint.py


Creating repo: blind-assist/internvl3_2b_walk_lora_8500_epoch3...
✅ Repo created (private)
📝 Generating Model Card...
✅ Model card saved
📤 Uploading full checkpoint from work_dirs/internvl3_2b_walk_lora...
   This may take a while due to large file sizes...
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...lk_lora/model.safetensors:   1%|▏             | 50.1MB / 4.47GB            

Processing Files (0 / 1)      :   1%|▏             | 50.1MB / 4.47GB,  125MB/s  

Processing Files (0 / 1)      :   2%|▎             | 92.0MB / 4.47GB,  154MB/s  

Processing Files (0 / 1)      :   3%|▍             |  134MB / 4.47GB,  168MB/s  

Processing Files (0 / 1)      :   4%|▋             |  201MB / 4.47GB,  201MB/s  

Processing Files (0 / 1)      :   6%|▊             |  251MB / 4.47GB,  210MB/s  

Processing Files (0 / 1)      :   7%|▉             |  310MB / 4.47GB,  222MB/s  

Proce

In [24]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 242.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [27]:
!apt-get update
!apt-get install -y ffmpeg

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1301 kB]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6538 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2378 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3737 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [39.2 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:12 http://archive.u

In [29]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.10.0
Uninstalling torchcodec-0.10.0:
  Successfully uninstalled torchcodec-0.10.0


In [31]:
!python test_hf_dataset.py --max_samples 900

🔄 Loading base model: OpenGVLab/InternVL3-2B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
🔄 Loading checkpoint weights from: work_dirs/internvl3_2b_walk_lora
   Loading: model.safetensors
   📊 Total checkpoint tensors: 1077

   ✅ Successfully merged 196 LoRA layers
   ✅ Total weights loaded: 685
✅ Model loaded with MERGED LoRA fine-tuned weights!

📁 Loading H

In [24]:
ls -F /workspace/InternVL/internvl_chat/

'=0.43.0'                            test_finetuned_model.py
 README.md                           test_from_huggingface.py
 data/                               tools/
 eval/                               train_walk_lora.sh*
 evaluate.sh                         training_log.txt
 examples/                           upload_to_hf.py
 inference_results_base/             wandb/
 inference_results_finetuned/        work_dirs/
 inference_results_hf/               zero_stage1_config.json
 intern_vl_finetune.ipynb            zero_stage2_config.json
 internvl/                           zero_stage3_config.json
 my_videos/                          zero_stage3_config_100b.json
 prepare_walk_data.py                zero_stage3_config_100b_1e7_offload.json
 pyproject.toml                      zero_stage3_config_100b_1e8.json
'python upload_full_checkpoint.py'   zero_stage3_config_34b.json
 shell/                              zero_stage3_config_70b.json


In [18]:
# Cell: Check uploaded model on HuggingFace
from huggingface_hub import hf_hub_download, list_repo_files

repo_id = "blind-assist/internvl3-2b-walk-lora-v1"

print("📁 Files in HuggingFace repo:")
files = list_repo_files(repo_id)
for f in files:
    print(f"   {f}")

# Download and check adapter_config
config_path = hf_hub_download(repo_id, "adapter_config.json")
import json
with open(config_path) as f:
    config = json.load(f)
print("\n📄 adapter_config.json:")
print(json.dumps(config, indent=2))

📁 Files in HuggingFace repo:
   .gitattributes
   README.md
   adapter_config.json
   adapter_model.safetensors

📄 adapter_config.json:
{
  "auto_mapping": null,
  "base_model_name_or_path": "OpenGVLab/InternVL3-2B",
  "bias": "none",
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layers_pattern": null,
  "layers_to_transform": null,
  "lora_alpha": 128,
  "lora_dropout": 0.0,
  "modules_to_save": null,
  "peft_type": "LORA",
  "r": 128,
  "revision": null,
  "target_modules": [
    "up_proj",
    "o_proj",
    "k_proj",
    "down_proj",
    "v_proj",
    "gate_proj",
    "q_proj"
  ],
  "task_type": "CAUSAL_LM"
}


In [19]:
# Cell: Check local checkpoint details
%cd /workspace/InternVL/internvl_chat

import os
import glob
from safetensors.torch import load_file

LOCAL_DIR = "work_dirs/internvl3_2b_walk_lora"

print("📁 Local checkpoint files:")
!ls -la {LOCAL_DIR}/*.safetensors 2>/dev/null || echo "No safetensors in main dir"
!ls -la {LOCAL_DIR}/checkpoint-*/*.safetensors 2>/dev/null | head -10

# Count LoRA weights locally
safetensor_files = glob.glob(os.path.join(LOCAL_DIR, "model*.safetensors"))
if not safetensor_files:
    # Check best checkpoint
    safetensor_files = glob.glob(os.path.join(LOCAL_DIR, "checkpoint-*", "model*.safetensors"))

total_lora = 0
for sf in safetensor_files[:1]:  # Check first file
    weights = load_file(sf)
    lora_keys = [k for k in weights.keys() if 'lora' in k.lower()]
    total_lora += len(lora_keys)
    print(f"\n📊 {os.path.basename(sf)}:")
    print(f"   Total tensors: {len(weights)}")
    print(f"   LoRA tensors: {len(lora_keys)}")
    if lora_keys:
        print(f"   Sample LoRA keys: {lora_keys[:3]}")

/workspace/InternVL/internvl_chat
📁 Local checkpoint files:
-rw-rw-rw- 1 root root 4473507792 Feb 18 10:45 work_dirs/internvl3_2b_walk_lora/model.safetensors
-rw-rw-rw- 1 root root 4473507792 Feb 18 10:33 work_dirs/internvl3_2b_walk_lora/checkpoint-15/model.safetensors
-rw-rw-rw- 1 root root 4473507792 Feb 18 10:37 work_dirs/internvl3_2b_walk_lora/checkpoint-30/model.safetensors
-rw-rw-rw- 1 root root 4473507792 Feb 18 10:40 work_dirs/internvl3_2b_walk_lora/checkpoint-45/model.safetensors
-rw-rw-rw- 1 root root 4473507792 Feb 18 10:43 work_dirs/internvl3_2b_walk_lora/checkpoint-60/model.safetensors

📊 model.safetensors:
   Total tensors: 1077
   LoRA tensors: 392
   Sample LoRA keys: ['language_model.base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'language_model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight']


In [20]:
# Cell: Compare local vs HuggingFace weights
import torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

# Load HuggingFace weights
hf_weights_path = hf_hub_download("blind-assist/internvl3-2b-walk-lora-v1", "adapter_model.safetensors")
hf_weights = load_file(hf_weights_path)

print("📊 HuggingFace adapter weights:")
print(f"   Total tensors: {len(hf_weights)}")
print(f"   Sample keys: {list(hf_weights.keys())[:5]}")

# Calculate weight statistics
for key in list(hf_weights.keys())[:3]:
    w = hf_weights[key]
    print(f"   {key}: shape={w.shape}, mean={w.float().mean():.6f}, std={w.float().std():.6f}")

📊 HuggingFace adapter weights:
   Total tensors: 392
   Sample keys: ['base_model.model.language_model.model.layers.0.mlp.down_proj.lora_A.weight', 'base_model.model.language_model.model.layers.0.mlp.down_proj.lora_B.weight', 'base_model.model.language_model.model.layers.0.mlp.gate_proj.lora_A.weight', 'base_model.model.language_model.model.layers.0.mlp.gate_proj.lora_B.weight', 'base_model.model.language_model.model.layers.0.mlp.up_proj.lora_A.weight']
   base_model.model.language_model.model.layers.0.mlp.down_proj.lora_A.weight: shape=torch.Size([128, 8960]), mean=0.000002, std=0.006108
   base_model.model.language_model.model.layers.0.mlp.down_proj.lora_B.weight: shape=torch.Size([1536, 128]), mean=0.000000, std=0.000267
   base_model.model.language_model.model.layers.0.mlp.gate_proj.lora_A.weight: shape=torch.Size([128, 1536]), mean=-0.000043, std=0.014747
